# Tracing Refusal Circuits in Gemma-2-2B

This notebook uses [circuit-tracer](https://github.com/decoderesearch/circuit-tracer) to investigate how Gemma-2-2B internally represents refusal behaviour. We:

1. Generate attribution graphs for structurally matched harmful/benign prompt pairs
2. Identify transcoder features that appear only in harmful-prompt graphs (candidate refusal features)
3. Check which candidate features are consistent across prompt categories
4. Ablate candidate features and observe whether the model stops refusing

In [ ]:
# Install dependencies (run once)
# !git clone https://github.com/decoderesearch/circuit-tracer.git
# !pip install ./circuit-tracer
# !pip install transformers torch

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
from circuit_tracer import ReplacementModel
from circuit_tracer.attribution import attribute
from src.prompts import PROMPT_PAIRS
from src.analysis import (
    extract_top_features,
    compare_graphs,
    find_consistent_refusal_features,
    ablation_experiment,
)

## 1. Load model with transcoders

We use Gemma-2-2B with GemmaScope PLT transcoders. This fits within Colab's free 15GB GPU.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = ReplacementModel.from_pretrained(
    "google/gemma-2-2b",
    transcoders="mntss/gemma-scope-transcoders",
    dtype=torch.float16,
    offload="cpu",
    device=device,
)
print(f"Model loaded on {device}")

## 2. Generate attribution graphs for each prompt pair

For each pair, we trace both the harmful and benign prompt and extract the top features.

In [ ]:
all_results = []

for pair in PROMPT_PAIRS:
    print(f"\n--- {pair['category']} ---")
    print(f"  Harmful: {pair['harmful'][:60]}...")
    print(f"  Benign:  {pair['benign'][:60]}...")

    graph_harmful = attribute(
        model=model,
        prompt=pair["harmful"],
        max_n_logits=10,
    )

    graph_benign = attribute(
        model=model,
        prompt=pair["benign"],
        max_n_logits=10,
    )

    harmful_features = extract_top_features(graph_harmful, top_k=50)
    benign_features = extract_top_features(graph_benign, top_k=50)

    result = compare_graphs(harmful_features, benign_features)
    result["category"] = pair["category"]
    result["prompts"] = pair
    all_results.append(result)

    n_refusal = len(result["refusal_candidates"])
    n_shared = len(result["shared"])
    print(f"  Refusal candidates: {n_refusal}, Shared: {n_shared}")

## 3. Find consistent refusal features across prompt categories

A feature is a strong refusal candidate if it appears in harmful-only sets across multiple prompt pairs.

In [ ]:
consistent = find_consistent_refusal_features(all_results, min_frequency=3)

print(f"Found {len(consistent)} features appearing in >= 3 harmful-only sets:\n")
for layer, feature_idx, count in consistent[:20]:
    print(f"  Layer {layer:2d}, Feature {feature_idx:6d} — appeared in {count}/{len(PROMPT_PAIRS)} pairs")

## 4. Intervention experiments

For the top consistent refusal features, ablate each one individually and observe whether the model stops refusing the harmful prompt.

In [ ]:
test_prompt = PROMPT_PAIRS[0]["harmful"]
top_candidates = [(layer, idx) for layer, idx, _ in consistent[:5]]

print(f"Test prompt: {test_prompt}\n")
print("=" * 80)

results = ablation_experiment(
    model,
    prompt=test_prompt,
    candidate_features=top_candidates,
    max_new_tokens=100,
)

for key, output in results.items():
    print(f"\n[{key}]")
    print(output)
    print("-" * 80)

## 5. Cross-category analysis

Check whether the top refusal features are shared across categories (violence, deception, illegal activity) or are category-specific.

In [ ]:
from collections import defaultdict

category_features = defaultdict(set)
for result in all_results:
    cat = result["category"]
    for feat in result["refusal_candidates"]:
        category_features[cat].add((feat.layer, feat.feature_idx))

categories = list(category_features.keys())
print("Feature overlap between categories:\n")
for i, cat_a in enumerate(categories):
    for cat_b in categories[i + 1:]:
        overlap = category_features[cat_a] & category_features[cat_b]
        print(f"  {cat_a} ∩ {cat_b}: {len(overlap)} shared refusal features")

universal = set.intersection(*category_features.values()) if category_features else set()
print(f"\nUniversal refusal features (all categories): {len(universal)}")
for layer, idx in sorted(universal):
    print(f"  Layer {layer}, Feature {idx}")

## 6. Amplification test

As a sanity check: amplifying refusal features on a *benign* prompt should cause the model to refuse even harmless requests.

In [ ]:
from src.analysis import run_intervention

benign_prompt = PROMPT_PAIRS[0]["benign"]
print(f"Benign prompt: {benign_prompt}\n")

baseline = model.generate(benign_prompt, max_new_tokens=100)
print(f"[baseline]\n{baseline}\n")

if consistent:
    layer, idx, _ = consistent[0]
    amplified = run_intervention(
        model, benign_prompt, layer, idx, scale=5.0, max_new_tokens=100
    )
    print(f"[amplified L{layer} F{idx} x5.0]\n{amplified}")